# 05 - Model Evaluation

This notebook evaluates the trained model on test set.

## Objectives
- Load trained model
- Generate predictions
- Calculate metrics (accuracy, precision, recall, F1, AUC)
- Create visualizations
- Analyze predictions

# Load model architecture
class MultimodalModel(nn.Module):
    def __init__(self, text_dim=768, image_dim=768, metadata_dim=32, hidden_dim=256):
        super().__init__()
        
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.image_proj = nn.Linear(image_dim, hidden_dim)
        self.metadata_proj = nn.Linear(metadata_dim, hidden_dim)
        
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3),
            nn.Softmax(dim=1)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    
    def forward(self, text, image, metadata):
        text_feat = torch.relu(self.text_proj(text))
        image_feat = torch.relu(self.image_proj(image))
        metadata_feat = torch.relu(self.metadata_proj(metadata))
        
        concat_feat = torch.cat([text_feat, image_feat, metadata_feat], dim=1)
        gates = self.gate(concat_feat)
        
        fused = (gates[:, 0:1] * text_feat + 
                 gates[:, 1:2] * image_feat + 
                 gates[:, 2:3] * metadata_feat)
        
        logits = self.classifier(fused)
        return logits

# Load embeddings
text_embeddings = np.load(EMBEDDINGS_DIR / 'text' / 'all_embeddings.npy')
image_embeddings = np.load(EMBEDDINGS_DIR / 'image' / 'all_embeddings.npy')
metadata_features = np.load(EMBEDDINGS_DIR / 'metadata' / 'all_features.npy')
id_mapping = pd.read_csv(EMBEDDINGS_DIR / 'id_mapping.csv')

# Create tensors
text_tensor = torch.from_numpy(text_embeddings).float().to(device)
image_tensor = torch.from_numpy(image_embeddings).float().to(device)
metadata_tensor = torch.from_numpy(metadata_features).float().to(device)
labels = torch.from_numpy(id_mapping['label'].values).float().to(device)

# Get test indices
test_idx = id_mapping[id_mapping['split'] == 'test'].index.values
print(f"Test set size: {len(test_idx)}")

## Generate Predictions

## Calculate Metrics

# Classification report
print("\nClassification Report:")
print(classification_report(test_labels, predictions_binary, target_names=['Normal', 'Misinformation']))

# Confusion matrix
cm = confusion_matrix(test_labels, predictions_binary)
print(f"\nConfusion Matrix:")
print(cm)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Misinformation'],
            yticklabels=['Normal', 'Misinformation'])
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'figures' / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Metrics bar plot
fig, ax = plt.subplots(figsize=(10, 6))
metric_names = list(metrics.keys())
metric_values = list(metrics.values())
bars = ax.bar(metric_names, metric_values, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6'])
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Evaluation Metrics', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, metric_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'figures' / 'metrics.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot prediction probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution by true class
for true_class in [0, 1]:
    mask = test_labels == true_class
    class_probs = predictions_probs[mask].flatten()
    class_name = 'Normal' if true_class == 0 else 'Misinformation'
    axes[0].hist(class_probs, bins=30, alpha=0.6, label=class_name)

axes[0].set_xlabel('Prediction Probability', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Prediction Probability Distribution by True Class', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
data_to_plot = [predictions_probs[test_labels == 0].flatten(),
                 predictions_probs[test_labels == 1].flatten()]
axes[1].boxplot(data_to_plot, labels=['Normal', 'Misinformation'])
axes[1].set_ylabel('Prediction Probability', fontsize=12)
axes[1].set_title('Prediction Probability Distribution', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'figures' / 'prediction_distribution.png', dpi=300, bbox_inches='tight')
plt.show()